
# Ionization parameter (logU) controls emission-line diagnostics

The ionization parameter logU controls the hardness of the ionizing
radiation field and drives rapid changes in optical line ratios. We show
how [OIII]/[OII] (O32) and [OIII]/Hβ respond to logU from -4 to -1 at
fixed metallicity (Z/Zsun = -0.5), demonstrating the use of O32 as a logU
diagnostic (Kewley & Dolphin 2002). Cue (Li et al. 2024, 2025) samples
the ionizing spectrum flexibility and provides smooth gradients through
metallicity, density, and ionization parameters for joint SED fitting.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")
warnings.filterwarnings("ignore", message=".*deprecated.*")

ssp = tengri.load_ssp("fsps_prsc_miles_chabrier")

# Build model with neb_logU as a free parameter
model = tengri.SEDModel.build(
    ssp,
    sfh={
        "type": "dpl",
        "*": tengri.FIXED,
        "alpha": 1.0,
        "beta": 2.5,
        "tau_gyr": 0.05,
        "log_total_mass": 10.0,
    },
    dust={"type": "two_component", "*": tengri.FIXED, "tau_diff": 0.0, "tau_bc": 0.0},
    neb={
        "type": "cue",
        "*": tengri.FIXED,
        "logZ_gas": -0.5,
        "logU": tengri.Uniform(-4.0, -1.0),
    },
    redshift=tengri.Fixed(0.05),
)

# Sample logU values and compute line ratios
logu_values = np.linspace(-4.0, -1.0, 15)
o32_ratio = []  # [OIII] / [OII]
o3hb_ratio = []  # [OIII] / Hbeta

# Get baseline parameters (fixed SFH, dust, redshift, metallicity)
baseline_params = dict(model.spec.sample(jax.random.PRNGKey(0)))

for logu in logu_values:
    # Modify only neb_logU, keep everything else fixed
    params = {**baseline_params, "neb_logU": np.float64(logu)}
    lines = model.predict_emission_lines(params)

    if lines is not None and not np.isnan(lines.oii):
        oii = float(lines.oii)
        oiii = float(lines.oiii_5007)
        hb = float(lines.hbeta)

        if oii > 0 and oiii > 0 and hb > 0:
            o32_ratio.append(np.log10(oiii / oii))
            o3hb_ratio.append(np.log10(oiii / hb))

# Create figure with single panel showing two traces
fig, ax = plt.subplots(figsize=(7, 5))

valid_logu = logu_values[: len(o32_ratio)]
ax.plot(
    valid_logu,
    o32_ratio,
    "o-",
    lw=2.0,
    ms=4,
    label=r"$\log([{\rm OIII}]/[{\rm OII}])$ (O32)",
    color="C0",
)
ax.plot(
    valid_logu,
    o3hb_ratio,
    "s-",
    lw=2.0,
    ms=4,
    label=r"$\log([{\rm OIII}]/{\rm H\beta})$",
    color="C1",
)

ax.set_xlabel(r"Ionization parameter $\log U$", fontsize=12)
ax.set_ylabel(r"Log line ratio", fontsize=12)
ax.legend(loc="upper left", fontsize=11, framealpha=0.95)
ax.grid(True, alpha=0.2)
ax.set_xlim(-4.1, -0.9)

fig.tight_layout()
plt.savefig("plot_cue_logu_line_ratios.png", dpi=150, bbox_inches="tight")
plt.close(fig)